In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "8"

In [ ]:
from pathlib import Path
import random
import shutil

ROOT = Path("./")
INPUT_IMAGES = ROOT / "input_images"
LABELS_SRC   = ROOT / "labels"

TRAIN_IMG = ROOT / "train/images"
TRAIN_LBL = ROOT / "train/labels"

VAL_IMG   = ROOT / "valid/images"
VAL_LBL   = ROOT / "valid/labels"

for p in [TRAIN_IMG, TRAIN_LBL, VAL_IMG, VAL_LBL]:
    p.mkdir(parents=True, exist_ok=True)

In [ ]:
label_files = list(LABELS_SRC.glob("*.txt"))

valid_pairs = []

for lbl in label_files:
    img = INPUT_IMAGES / (lbl.stem + ".jpg")
    if img.exists():
        valid_pairs.append((img, lbl))

print("유효한 (image, label) 쌍:", len(valid_pairs))


In [ ]:
label_files = list(LABELS_SRC.glob("*.txt"))

pairs = []
for lbl in label_files:
    img = INPUT_IMAGES / f"{lbl.stem}.jpg"
    if img.exists():
        pairs.append((img, lbl))

print("유효한 데이터 수:", len(pairs))


In [ ]:
# train : valid = 8:2
random.seed(42)
random.shuffle(valid_pairs)

split_idx = int(len(valid_pairs) * 0.8)
train_pairs = valid_pairs[:split_idx]
val_pairs   = valid_pairs[split_idx:]

print("Train:", len(train_pairs))
print("Val:", len(val_pairs))


In [ ]:
# file copy (image+label)

def copy_pairs(pairs, img_dst, lbl_dst):
    for img, lbl in pairs:
        shutil.copy(img, img_dst / img.name)
        shutil.copy(lbl, lbl_dst / lbl.name)

copy_pairs(train_pairs, TRAIN_IMG, TRAIN_LBL)
copy_pairs(val_pairs, VAL_IMG, VAL_LBL)

print("✔ train / valid 데이터 복사 완료")


In [ ]:
# 무결성 체크

train_imgs = {p.stem for p in TRAIN_IMG.glob("*.jpg")}
train_lbls = {p.stem for p in TRAIN_LBL.glob("*.txt")}

valid_imgs = {p.stem for p in VAL_IMG.glob("*.jpg")}
valid_lbls = {p.stem for p in VAL_LBL.glob("*.txt")}

print("Train mismatch:", train_imgs ^ train_lbls)
print("Valid mismatch:", valid_imgs ^ valid_lbls)
